# Stereo Visual Odometry Example (Large Dataset)

In this notebook we build on `StereoVOExample.ipynb` to tackle a more realistic stereo visual-odometry problem:  
A robot carries a stereo camera, moves forward over a longer trajectory, and observes dozens of landmarks. Our goals are:

1. Define a noise model and stereo calibration.  
2. Assemble a factor graph of stereo-measurement factors.  
3. Provide initial estimates for both camera poses and landmark positions.  
4. Run Levenberg–Marquardt optimization to recover the most likely trajectory and map.  
5. Analyze and visualize final poses and landmarks.


If any section may seem unclear, we recommend looking at `StereoVOExample.ipynb` first and coming back to this example afterwards. 

GTSAM Copyright 2010-2025, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/StereoVOExample_large.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries
%pip install -q gtsam-develop
%pip install -q numpy
%pip install -q matplotlib

In [1]:
import requests
from io import StringIO
import numpy as np
import gtsam
from gtsam.utils.plot import plot_pose3
import matplotlib.pyplot as plt
import gtsam.utils.plot as gtsam_plot

# For shorthand for common GTSAM types (like X for poses, L for landmarks)
from gtsam.symbol_shorthand import X, L

## 1. Setup Factor Graph

We start by creating an empty `NonlinearFactorGraph`.

In [2]:
# Create an empty nonlinear factor graph
graph = gtsam.NonlinearFactorGraph()

## 2. Define Camera Calibration and Measurement Noise
- **Noise Model:** We define an isotropic noise model for the stereo measurements. `Isotropic.Sigma(3, 1.0)` means a 3-dimensional noise (for $u_L, u_R, v$) with a standard deviation of 1 pixel for each dimension.
- **Camera Calibration ([`Cal3_S2Stereo`](/gtsam/geometry/doc/Cal3_S2Stereo.ipynb)):** We download the calibration parameters text file from the GTSAM Github and setup a stereo calibration model with those parameters:
  - $f_x=f_y=721.5377$
  - $s=0$
  - $(u,v)=(609.5593,\,172.854)$
  - $b=0.537150588$

In [3]:
# Retrieve Calibration file
url = "https://raw.githubusercontent.com/borglab/gtsam/refs/heads/develop/examples/Data/VO_calibration.txt"
response = requests.get(url)
cal_params_file = StringIO(response.text)

# Read calibration parameters
cal_params = np.loadtxt(cal_params_file)

# Create a Cal3_S2Stereo calibration object
K = gtsam.Cal3_S2Stereo(cal_params)
print(K)

# Define the stereo measurement noise model (isotropic, 1 pixel standard deviation)
measurement_noise_model = gtsam.noiseModel.Isotropic.Sigma(3, 1.0)

K: 721.538       0 609.559
      0 721.538 172.854
      0       0       1
Baseline: 0.537151



## 3. Load Camera Pose Data

The pose data is stored in a text file. Each row corresponds to a 3D pose and contains 17 values; the first value represents the ID of that pose, and the remaining 16 values represents the $4\times4$ pose matrix flattened according to row-major order. The following code downloads a file containing data of 26 different poses and loads them into a `gtsam.Values` object called `initial_estimate`, which functions effectively like a map or dictionary.

In [4]:
# Odometry data file with camera poses
url = "https://raw.githubusercontent.com/borglab/gtsam/refs/heads/develop/examples/Data/VO_camera_poses_large.txt"
response = requests.get(url)
odometry_data_file = StringIO(response.text)

# Read poses
camera_poses = np.loadtxt(odometry_data_file)

# Create a gtsam.Values object to hold the pose data
initial_estimates = gtsam.Values()

# Parse the raw camera pose data
for pose in camera_poses:
	pose_id = int(pose[0])
	m = pose[1:].copy().reshape(4, 4)		# 16 data values reshaped to a 4x4 matrix with row major order
	initial_estimates.insert(X(pose_id), gtsam.Pose3(m))

## 4. Load Stereo Pixel and Landmark Coordinates

The factor data file stores the pixel coordinates $(u_L, u_R, v)$ and landmark coordinates $(X, Y, Z)$ in the camera frame. Note that there is only one set of pixel coordinates per entry due image rectification, and the landmark coordinates are results from triangulation. Just like the pose data, each row, with 8 values, represents a distinct entry; the first two values are the pose ID and the landmark ID, respectively, the next triplet of values represent the pixel coordinates $(u_L, u_R, v)$, and the last triplet of values represent the landmark coordinates $(X, Y, Z)$.

In [5]:
# Factor data file with pixel coordinates and landmark coordinates
url = "https://raw.githubusercontent.com/borglab/gtsam/refs/heads/develop/examples/Data/VO_stereo_factors_large.txt"
response = requests.get(url)
factor_data_file = StringIO(response.text)

# Read factor data
data_matrix = np.loadtxt(factor_data_file)
for entry in data_matrix:
	pose_id = int(entry[0])
	landmark_id = int(entry[1])
	uL, uR, v = entry[2:5]
	land_X, land_Y, land_Z = entry[5:8]

	graph.add(gtsam.GenericStereoFactor3D(
		gtsam.StereoPoint2(uL, uR, v), measurement_noise_model, X(pose_id), L(landmark_id), K
	))
	# If the landmark variable included in this factor as not yet been added
	# to the initial variable value estimate, add it
	if not initial_estimates.exists(L(landmark_id)):
		cam_pose = initial_estimates.atPose3(X(pose_id))
		# transformFrom() transforms the input Point3 from the camera pose space
		# to the global space
		world_point = cam_pose.transformFrom(gtsam.Point3(land_X, land_Y, land_Z))
		initial_estimates.insert(L(landmark_id), world_point)

## 5. Constrain First Pose
We constrain the first pose such that it cannot change from its original value during optimization.

In [6]:
first_pose = initial_estimates.atPose3(X(1))

graph.add(gtsam.NonlinearEqualityPose3(X(1), first_pose))

## 6. Optimize the Factor Graph with Levenberg-Marquardt

We will use a `LevenbergMarquardtOptimizer` to optimize the factor graph. This will take around 10-20 seconds.

In [7]:
params = gtsam.LevenbergMarquardtParams()
params.setOrderingType("METIS")

optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimates, params)
result = optimizer.optimize()

## 7. Analyze Results

We can extract the optimized poses from the `result` Values object.

In [9]:
print("Final result sample:")

pose_values = gtsam.utilities.allPose3s(result)
print(pose_values)

Final result sample:
Values with 26 values:
Value x1: (gtsam::Pose3)
R: [
	1, 0, 0;
	0, 1, 0;
	0, 0, 1
]
t: 0 0 0

Value x2: (gtsam::Pose3)
R: [
	0.999997, -0.00234524, 0.000667659;
	0.00234572, 0.999997, -0.000714904;
	-0.000665981, 0.000716469, 1
]
t: 0.00254935 0.00430119   0.959176

Value x3: (gtsam::Pose3)
R: [
	0.999993, -0.00363586, 0.000733503;
	0.00363353, 0.999988, 0.00316275;
	-0.000744993, -0.00316007, 0.999995
]
t: 0.00112241 0.00989265     1.9179

Value x4: (gtsam::Pose3)
R: [
	0.999982, -0.00599989, 8.41316e-05;
	0.00599887, 0.99994, 0.00918689;
	-0.000139247, -0.00918622, 0.999958
]
t: 0.00127606  0.0117263    2.87189

Value x5: (gtsam::Pose3)
R: [
	0.999952, -0.00962861, -0.00135944;
	0.00964083, 0.999911, 0.00925616;
	0.0012702, -0.00926882, 0.999957
]
t: 0.00314102  0.0166897    3.81369

Value x6: (gtsam::Pose3)
R: [
	0.999922, -0.0122294, -0.00243049;
	0.012245, 0.999903, 0.00654308;
	0.00235024, -0.00657234, 0.999976
]
t: 0.00124453   0.021009    4.75581

Value x7: